In [2]:
from sqlalchemy import create_engine , text
from sqlalchemy.engine import URL
import pandas as pd
import random
from faker import Faker

In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

def get_engine():
    url = URL.create(
        drivername="mysql+pymysql",
        username=os.getenv("DB_USER"),
        password=os.getenv("DB_PASSWORD"),
        host=os.getenv("DB_HOST"),
        port=3306,
        database="Banking_analytics"
    )
    engine = create_engine(url)

    return engine

In [81]:
def get_query():
    query = """
        CREATE TABLE transactions (
        transaction_id INT AUTO_INCREMENT PRIMARY KEY,
        account_id INT NOT NULL,
        transaction_date DATE NOT NULL,
        transaction_type VARCHAR(20) NOT NULL,
        transaction_mode VARCHAR(30) NOT NULL,
        amount DECIMAL(12,2) NOT NULL,
        balance_after_transaction DECIMAL(12,2) NOT NULL,
        description VARCHAR(100) NOT NULL,
        FOREIGN KEY (account_id) REFERENCES accounts(account_id)
    );
    """
    return query

In [83]:
query = get_query()
engine = get_engine()
with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()
print("complited")

In [ ]:
pd.read_sql("SHOW TABLES;", engine)


In [ ]:
query = """
SELECT
    account_id,
    customer_id,
    account_type,
    account_balance,
    account_opening_date
FROM accounts;
"""

df = pd.read_sql(query, engine)

In [ ]:
df.shape


In [ ]:
df["num_transactions"] = [
    random.randint(5, 10)
    for _ in range(len(df))
]

In [ ]:
df["num_transactions"].value_counts().sort_index()

In [ ]:
df["num_transactions"].sum()

In [ ]:
transaction_types = [
    "Credit",
    "Debit"
]

transaction_modes = [
    "UPI",
    "ATM",
    "Debit Card",
    "Credit Card",
    "NEFT",
    "RTGS",
    "IMPS",
    "Cash Deposit",
    "Cash Withdrawal",
    "Cheque"
]

descriptions = [
    "Salary",
    "Shopping",
    "Electricity Bill",
    "Water Bill",
    "Mobile Recharge",
    "Restaurant",
    "Fuel",
    "ATM Withdrawal",
    "Cash Deposit",
    "Online Purchase",
    "EMI Payment",
    "Rent",
    "Insurance",
    "Interest Credit",
    "Fund Transfer"
]

In [84]:
print(transaction_types)
print(transaction_modes)
print(descriptions)

['Credit', 'Debit']
['UPI', 'ATM', 'Debit Card', 'Credit Card', 'NEFT', 'RTGS', 'IMPS', 'Cash Deposit', 'Cash Withdrawal', 'Cheque']
['Salary', 'Shopping', 'Electricity Bill', 'Water Bill', 'Mobile Recharge', 'Restaurant', 'Fuel', 'ATM Withdrawal', 'Cash Deposit', 'Online Purchase', 'EMI Payment', 'Rent', 'Insurance', 'Interest Credit', 'Fund Transfer']


In [85]:
def generate_amount(transaction_type, current_balance):

    if transaction_type == "Credit":
        return random.randint(500, 100000)

    # Agar balance bahut kam hai to debit mat hone do
    if current_balance < 500:
        return 0

    max_debit = min(
        int(current_balance * 0.4),
        50000,
        int(current_balance)
    )

    if max_debit < 500:
        return random.randint(1, max(1, max_debit))

    return random.randint(500, max_debit)

In [86]:
print(generate_amount("Credit", 50000))
print(generate_amount("Debit", 50000))

80527
11605


In [87]:
sample_account = df.iloc[0]

sample_account

account_id                       1
customer_id                      1
account_type                Saving
account_balance          3860000.0
account_opening_date    2023-01-11
num_transactions                 6
Name: 0, dtype: object

In [88]:
current_balance = sample_account["account_balance"]

transactions = []

for _ in range(sample_account["num_transactions"]):

    transaction_type = random.choice(transaction_types)

    amount = generate_amount(
        transaction_type,
        current_balance
    )

    if transaction_type == "Credit":
        current_balance += amount
    else:
        current_balance -= amount

    transactions.append({
        "account_id": sample_account["account_id"],
        "transaction_type": transaction_type,
        "amount": amount,
        "balance_after_transaction": round(current_balance, 2)
    })

In [89]:
pd.DataFrame(transactions)

,account_id,transaction_type,amount,balance_after_transaction
0,1,Debit,33227,3826773.0
1,1,Debit,38917,3787856.0
2,1,Credit,94376,3882232.0
3,1,Credit,46120,3928352.0
4,1,Credit,66337,3994689.0
5,1,Credit,17316,4012005.0


In [90]:
credit_transactions = {
    "Salary": ["NEFT", "IMPS"],
    "Interest Credit": ["IMPS"],
    "Cash Deposit": ["Cash Deposit"],
    "Fund Transfer": ["UPI", "NEFT", "IMPS"]
}

debit_transactions = {
    "Shopping": ["Debit Card", "UPI"],
    "Restaurant": ["UPI", "Credit Card"],
    "Electricity Bill": ["UPI"],
    "Water Bill": ["UPI"],
    "Mobile Recharge": ["UPI"],
    "Fuel": ["Debit Card"],
    "ATM Withdrawal": ["ATM"],
    "EMI Payment": ["NEFT"],
    "Rent": ["UPI", "NEFT"],
    "Insurance": ["NEFT"],
    "Online Purchase": ["Debit Card", "Credit Card"]
}

In [91]:
print(credit_transactions)
print(debit_transactions)

{'Salary': ['NEFT', 'IMPS'], 'Interest Credit': ['IMPS'], 'Cash Deposit': ['Cash Deposit'], 'Fund Transfer': ['UPI', 'NEFT', 'IMPS']}
{'Shopping': ['Debit Card', 'UPI'], 'Restaurant': ['UPI', 'Credit Card'], 'Electricity Bill': ['UPI'], 'Water Bill': ['UPI'], 'Mobile Recharge': ['UPI'], 'Fuel': ['Debit Card'], 'ATM Withdrawal': ['ATM'], 'EMI Payment': ['NEFT'], 'Rent': ['UPI', 'NEFT'], 'Insurance': ['NEFT'], 'Online Purchase': ['Debit Card', 'Credit Card']}


In [92]:
def generate_transaction(current_balance):

    transaction_type = random.choices(
        ["Credit", "Debit"],
        weights=[30, 70],
        k=1
    )[0]

    if transaction_type == "Credit":

        description = random.choices(
            ["Salary", "Fund Transfer", "Cash Deposit", "Interest Credit"],
            weights=[20, 50, 20, 10],
            k=1
        )[0]

        mode = random.choice(credit_transactions[description])

    else:

        description = random.choices(
            [
                "Shopping",
                "Restaurant",
                "Electricity Bill",
                "Water Bill",
                "Mobile Recharge",
                "Fuel",
                "ATM Withdrawal",
                "EMI Payment",
                "Rent",
                "Insurance",
                "Online Purchase"
            ],
            weights=[20, 10, 5, 3, 8, 10, 5, 8, 8, 3, 20],
            k=1
        )[0]

        mode = random.choice(debit_transactions[description])

    amount = generate_amount(transaction_type, current_balance)

    if transaction_type == "Credit":
        current_balance += amount
    else:
        current_balance -= amount

    return {
        "transaction_type": transaction_type,
        "transaction_mode": mode,
        "description": description,
        "amount": amount,
        "balance_after_transaction": round(current_balance, 2)
    }

In [93]:
balance = 100000

for i in range(5):

    tx = generate_transaction(balance)

    balance = tx["balance_after_transaction"]

    print(tx)

{'transaction_type': 'Credit', 'transaction_mode': 'IMPS', 'description': 'Interest Credit', 'amount': 22048, 'balance_after_transaction': 122048}
{'transaction_type': 'Debit', 'transaction_mode': 'UPI', 'description': 'Restaurant', 'amount': 24959, 'balance_after_transaction': 97089}
{'transaction_type': 'Debit', 'transaction_mode': 'Debit Card', 'description': 'Fuel', 'amount': 20924, 'balance_after_transaction': 76165}
{'transaction_type': 'Credit', 'transaction_mode': 'IMPS', 'description': 'Fund Transfer', 'amount': 92967, 'balance_after_transaction': 169132}
{'transaction_type': 'Debit', 'transaction_mode': 'Credit Card', 'description': 'Online Purchase', 'amount': 45063, 'balance_after_transaction': 124069}


In [94]:
from datetime import datetime, timedelta

all_transactions = []

start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)

for _, row in df.iterrows():

    current_balance = float(row["account_balance"])

    account_id = row["account_id"]

    for _ in range(row["num_transactions"]):

        tx = generate_transaction(current_balance)

        current_balance = tx["balance_after_transaction"]

        random_days = random.randint(
            0,
            (end_date - start_date).days
        )

        transaction_date = (
            start_date + timedelta(days=random_days)
        ).date()

        all_transactions.append({

            "account_id": account_id,

            "transaction_date": transaction_date,

            "transaction_type": tx["transaction_type"],

            "transaction_mode": tx["transaction_mode"],

            "amount": tx["amount"],

            "balance_after_transaction": tx["balance_after_transaction"],

            "description": tx["description"]

        })

In [95]:
transactions_df = pd.DataFrame(all_transactions)

In [96]:
transactions_df.shape

(749024, 7)

In [97]:
transactions_df.head()

,account_id,transaction_date,transaction_type,transaction_mode,amount,balance_after_transaction,description
0,1,2024-10-11,Credit,IMPS,82329,3942329.0,Interest Credit
1,1,2022-03-23,Debit,NEFT,46183,3896146.0,EMI Payment
2,1,2023-04-25,Debit,Debit Card,9483,3886663.0,Fuel
3,1,2025-12-17,Credit,UPI,18578,3905241.0,Fund Transfer
4,1,2023-06-05,Credit,NEFT,86901,3992142.0,Fund Transfer


In [98]:
transactions_df = transactions_df.sort_values(
    by=["account_id", "transaction_date"]
).reset_index(drop=True)

In [99]:
transactions_df.head(10)

,account_id,transaction_date,transaction_type,transaction_mode,amount,balance_after_transaction,description
0,1,2022-03-23,Debit,NEFT,46183,3896146.0,EMI Payment
1,1,2023-04-25,Debit,Debit Card,9483,3886663.0,Fuel
2,1,2023-06-05,Credit,NEFT,86901,3992142.0,Fund Transfer
3,1,2024-07-17,Credit,IMPS,72841,4064983.0,Fund Transfer
4,1,2024-10-11,Credit,IMPS,82329,3942329.0,Interest Credit
5,1,2025-12-17,Credit,UPI,18578,3905241.0,Fund Transfer
6,2,2020-01-28,Debit,UPI,1057,2459.0,Shopping
7,2,2020-02-18,Debit,Debit Card,1160,3516.0,Online Purchase
8,2,2020-06-25,Debit,UPI,2189,6071.0,Electricity Bill
9,2,2020-08-24,Debit,UPI,501,1154.0,Shopping


In [100]:
account_balance_map = (
    df.set_index("account_id")["account_balance"]
      .astype(float)
      .to_dict()
)

In [101]:

new_balances = []
current_balance = account_balance_map.copy()

In [102]:
for _, row in transactions_df.iterrows():

    acc = row["account_id"]
    balance = current_balance[acc]

    if row["transaction_type"] == "Credit":
        balance += row["amount"]
    else:
        balance -= row["amount"]

    balance = round(balance, 2)

    current_balance[acc] = balance

    new_balances.append(balance)

In [103]:
transactions_df["balance_after_transaction"] = new_balances


In [104]:
transactions_df[
    transactions_df["account_id"] == 1
]

,account_id,transaction_date,transaction_type,transaction_mode,amount,balance_after_transaction,description
0,1,2022-03-23,Debit,NEFT,46183,3813817.0,EMI Payment
1,1,2023-04-25,Debit,Debit Card,9483,3804334.0,Fuel
2,1,2023-06-05,Credit,NEFT,86901,3891235.0,Fund Transfer
3,1,2024-07-17,Credit,IMPS,72841,3964076.0,Fund Transfer
4,1,2024-10-11,Credit,IMPS,82329,4046405.0,Interest Credit
5,1,2025-12-17,Credit,UPI,18578,4064983.0,Fund Transfer


In [105]:
(transactions_df["balance_after_transaction"] < 0).sum()

np.int64(29268)

In [106]:
transactions_df[
    transactions_df["balance_after_transaction"] < 0
].head(10)

,account_id,transaction_date,transaction_type,transaction_mode,amount,balance_after_transaction,description
117,16,2020-10-07,Debit,ATM,34401,-18401.0,ATM Withdrawal
118,16,2020-11-15,Debit,UPI,27626,-46027.0,Shopping
226,31,2020-04-27,Debit,Credit Card,7043,-4543.0,Online Purchase
227,31,2020-06-13,Debit,NEFT,1232,-5775.0,EMI Payment
253,35,2023-04-10,Debit,Debit Card,21040,-10070.0,Online Purchase
254,35,2023-07-20,Debit,NEFT,13232,-23302.0,EMI Payment
255,35,2024-09-01,Credit,IMPS,6101,-17201.0,Salary
268,38,2021-02-20,Debit,UPI,14840,-2340.0,Rent
269,38,2021-04-20,Debit,UPI,27883,-30223.0,Mobile Recharge
270,38,2021-05-12,Debit,UPI,12411,-42634.0,Electricity Bill


In [107]:
df[df["account_id"] == 2][["account_id", "account_balance"]]


,account_id,account_balance
1,2,18500.0


In [108]:
# Agar current_balance dictionary ban gaya hai to usse reset karo
if isinstance(current_balance, dict):
    current_balance = float(df.loc[df["account_id"] == 1, "account_balance"].iloc[0])

# Test
tx = generate_transaction(current_balance)
print(tx)

{'transaction_type': 'Credit', 'transaction_mode': 'NEFT', 'description': 'Fund Transfer', 'amount': 37455, 'balance_after_transaction': 3897455.0}


In [109]:
all_transactions = []

start_date = datetime(2020, 1, 1)
end_date = datetime(2025, 12, 31)

for _, row in df.iterrows():

    account_id = row["account_id"]
    opening_balance = float(row["account_balance"])
    n = int(row["num_transactions"])

    # Random dates generate karke sort kar do
    transaction_dates = sorted([
        start_date + timedelta(
            days=random.randint(0, (end_date - start_date).days)
        )
        for _ in range(n)
    ])

    current_balance = opening_balance

    for tx_date in transaction_dates:

        tx = generate_transaction(current_balance)

        current_balance = tx["balance_after_transaction"]

        all_transactions.append({
            "account_id": account_id,
            "transaction_date": tx_date.date(),
            "transaction_type": tx["transaction_type"],
            "transaction_mode": tx["transaction_mode"],
            "amount": tx["amount"],
            "balance_after_transaction": current_balance,
            "description": tx["description"]
        })

transactions_df = pd.DataFrame(all_transactions)

In [110]:
(transactions_df["balance_after_transaction"] < 0).sum()

np.int64(0)

In [111]:
transactions_df.to_sql(
    "transactions",
    con=engine,
    if_exists="append",
    index=False,
    chunksize=5000,
    method="multi"
)

749024